# 🪸 Servidor CoralSCOP na Nuvem (Google Colab GPU)
### Pipeline Híbrido: CoralSCOP ViT-B (CVPR 2024) + Pré-processamento Subaquático + CoralNet

Este notebook executa o **CoralSCOP (`vit_b_coralscop.pth`, 382 MB)** — o primeiro *Foundation Model* treinado especificamente com **+41.000 fotos de recifes e +330.000 máscaras de corais**.

**Por que CoralSCOP em vez do SAM comum?**
- Possui um **ramo semântico paralelo** no decodificador que segmenta **apenas colônias de corais** e ignora automaticamente canos de PVC, adesivos de calibração, sensores/fitas, fios de nylon, areia e rocha.
- **6,5x mais leve e ~2,5x mais rápido** na GPU Tesla T4 do que o SAM ViT-H (2.5 GB).
- Inclui **fechamento morfológico anti-fios** e **ordenação de múltiplos contornos por área**.

In [ ]:
# 1. Verificar GPU alocada pelo Google Colab
!nvidia-smi

In [ ]:
# 2. Instalar dependências, clonar repositório oficial do CoralSCOP e aplicar patches p/ PyTorch 2.6+
import os
import subprocess

# Remover segment-anything genérico da Meta caso exista na sessão para não conflitar com o CoralSCOP
os.system("pip uninstall -y segment-anything > /dev/null 2>&1")
!pip install -q fastapi uvicorn pyngrok nest_asyncio python-multipart pillow opencv-python pycocotools huggingface_hub

REPO_DIR = "/content/CoralSCOP"
if not os.path.exists(REPO_DIR):
    print("Clonando repositório oficial do CoralSCOP...")
    !git clone -q https://github.com/zhengziqiang/CoralSCOP.git {REPO_DIR}

# Patch 1: Compatibilidade com PyTorch 2.6+ (weights_only=False e map_location)
build_sam_path = os.path.join(REPO_DIR, "segment_anything", "build_sam.py")
with open(build_sam_path, "r", encoding="utf-8") as f:
    code_bs = f.read()
if "weights_only" not in code_bs:
    code_bs = code_bs.replace(
        "torch.load(f)",
        "torch.load(f, map_location='cpu', weights_only=False)"
    )
    with open(build_sam_path, "w", encoding="utf-8") as f:
        f.write(code_bs)

# Patch 2: Evitar erro dimensional quando um sub-quadrante de zoom não contém nenhum coral
amg_path = os.path.join(REPO_DIR, "segment_anything", "automatic_mask_generator.py")
with open(amg_path, "r", encoding="utf-8") as f:
    code_amg = f.read()
old_crop_line = 'data["crop_boxes"] = torch.tensor([crop_box for _ in range(len(data["rles"]))])'
new_crop_line = 'data["crop_boxes"] = torch.tensor([crop_box for _ in range(len(data["rles"]))], dtype=torch.long).reshape(-1, 4)'
if old_crop_line in code_amg:
    code_amg = code_amg.replace(old_crop_line, new_crop_line)
old_fc_block = 'fc_features=F.normalize(data["fc_features"],dim=-1)\n        similarity=fc_features.mm(fc_features.t())\n        similarity.fill_diagonal_(-np.inf)\n        data["similarity"]=similarity'
new_fc_block = (
    'if "fc_features" in data._stats and len(data["fc_features"]) > 0:\n'
    '            fc_features = F.normalize(data["fc_features"], dim=-1)\n'
    '            similarity = fc_features.mm(fc_features.t())\n'
    '            similarity.fill_diagonal_(-np.inf)\n'
    '            data["similarity"] = similarity\n'
    '        else:\n'
    '            data["similarity"] = torch.empty((0, 0))'
)
if old_fc_block in code_amg:
    code_amg = code_amg.replace(old_fc_block, new_fc_block)
with open(amg_path, "w", encoding="utf-8") as f:
    f.write(code_amg)

print("✅ Ambiente CoralSCOP preparado e corrigido com sucesso!")

In [ ]:
# 3. Baixar os pesos oficiais do CoralSCOP (vit_b_coralscop.pth - 382 MB)
import os
import urllib.request
from huggingface_hub import hf_hub_download

os.makedirs("/content/CoralSCOP/checkpoints", exist_ok=True)
WEIGHTS_FILE = "/content/CoralSCOP/checkpoints/vit_b_coralscop.pth"

if not os.path.exists(WEIGHTS_FILE) or os.path.getsize(WEIGHTS_FILE) < 350_000_000:
    print("Baixando vit_b_coralscop.pth (382 MB) via Hugging Face CDN...")
    try:
        downloaded = hf_hub_download(
            repo_id="reefsupport/CoralSCOP",
            filename="models/vit_b_coralscop.pth",
            local_dir="/content/coralscop_hf"
        )
        if os.path.lexists(WEIGHTS_FILE):
            os.remove(WEIGHTS_FILE)
        os.symlink(downloaded, WEIGHTS_FILE)
        print("✅ Download concluído via Hugging Face em alta velocidade!")
    except Exception as e:
        print(f"Aviso Hugging Face ({e}). Baixando do mirror oficial Dropbox...")
        DROPBOX_URL = "https://www.dropbox.com/scl/fi/pw5jiq9oc8e8kvkx1fdk0/vit_b_coralscop.pth?rlkey=qczdohnzxwgwoadpzeht0lim2&st=actcedwy&dl=1"
        urllib.request.urlretrieve(DROPBOX_URL, WEIGHTS_FILE)
        print("✅ Download concluído via mirror oficial!")
else:
    print("✅ Pesos do CoralSCOP (vit_b_coralscop.pth) já presentes no ambiente.")

In [ ]:
# 4. Carregar o CoralSCOP ViT-B na GPU
import sys
import torch

# Garantir que o pacote segment_anything carregado seja o do CoralSCOP (com ramo semântico de corais)
for mod_name in list(sys.modules.keys()):
    if mod_name == "segment_anything" or mod_name.startswith("segment_anything."):
        del sys.modules[mod_name]
if "/content/CoralSCOP" not in sys.path:
    sys.path.insert(0, "/content/CoralSCOP")

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Carregando CoralSCOP ViT-B em: {device} ({gpu_label})...")

coralscop_model = sam_model_registry["vit_b"](checkpoint=WEIGHTS_FILE)
coralscop_model.to(device=device)
coralscop_model.eval()

print("🪸 CoralSCOP ViT-B carregado e pronto para segmentar exclusivamente colônias de corais!")

In [ ]:
# 5. Criar a API FastAPI para receber as imagens pré-processadas da sua interface web
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import io
import cv2
import numpy as np
from pycocotools import mask as mask_utils

app = FastAPI(title="CoralSCOP Benthic Coral Segmentation Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {
        "status": "online",
        "model": "CoralSCOP ViT-B (CVPR 2024)",
        "engine": "coralscop",
        "device": device,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    }

@app.post("/segment")
async def segment_image(
    file: UploadFile = File(...),
    min_area: int = 100,
    pred_iou_thresh: float = 0.72,
    points_per_side: int = 32,
    stability_score_thresh: float = 0.62,
    crop_n_layers: int = 1,
    crop_n_points_downscale_factor: int = 2,
    bridge_strings: bool = True
):
    generator = SamAutomaticMaskGenerator(
        model=coralscop_model,
        points_per_side=int(points_per_side),
        pred_iou_thresh=float(pred_iou_thresh),
        stability_score_thresh=float(stability_score_thresh),
        crop_n_layers=int(crop_n_layers),
        crop_n_points_downscale_factor=int(crop_n_points_downscale_factor),
        min_mask_region_area=max(20, int(min_area))
    )

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")
    image_np = np.array(image)
    h, w, _ = image_np.shape

    try:
        raw_masks = generator.generate(image_np)
    except (KeyError, IndexError) as empty_err:
        print(f"[Aviso] Nenhuma colônia detectada na imagem ({empty_err}).")
        raw_masks = []

    # Kernel elíptico suave (7x7 em 1024px) para unir metades de colônias cortadas por fios finos de nylon
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    masks_output = []
    for idx, m in enumerate(raw_masks, 1):
        seg_uint8 = (m["segmentation"].astype(np.uint8)) * 255
        if bridge_strings:
            seg_uint8 = cv2.morphologyEx(seg_uint8, cv2.MORPH_CLOSE, close_kernel)

        contours, _ = cv2.findContours(seg_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Ordenar contornos do MAIOR para o MENOR (corrige bug onde fragmentos pequenos descartavam a colônia)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)

        simplified_polys = []
        for cnt in contours:
            if cv2.contourArea(cnt) < 20:
                continue
            approx = cv2.approxPolyDP(cnt, epsilon=1.4, closed=True)
            if len(approx) >= 3:
                simplified_polys.append(approx.reshape(-1, 2).tolist())

        if not simplified_polys:
            continue

        area = int(np.count_nonzero(seg_uint8))
        ys, xs = np.nonzero(seg_uint8)
        bx0, by0 = int(xs.min()), int(ys.min())
        bx1, by1 = int(xs.max()), int(ys.max())
        bbox = [bx0, by0, max(1, bx1 - bx0), max(1, by1 - by0)]

        # Centroide real dentro do maior contorno da colônia
        M = cv2.moments(contours[0])
        if M["m00"] > 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        else:
            cx = bx0 + bbox[2] // 2
            cy = by0 + bbox[3] // 2

        rle = mask_utils.encode(np.asfortranarray(seg_uint8 > 0))
        rle_dict = {"size": rle["size"], "counts": rle["counts"].decode("utf-8")}

        masks_output.append({
            "id": len(masks_output) + 1,
            "bbox": bbox,
            "centroid": [cy, cx],
            "area": area,
            "area_pct": (area / float(w * h)) * 100.0,
            "polygon": simplified_polys[0],
            "polygons": simplified_polys,
            "rle": rle_dict,
            "predicted_iou": float(m["predicted_iou"]),
            "stability_score": float(m["stability_score"])
        })

    return {
        "model": "CoralSCOP ViT-B (CVPR 2024)",
        "engine": "coralscop",
        "image_width": w,
        "image_height": h,
        "total_masks": len(masks_output),
        "masks": masks_output
    }


In [ ]:
# @title 6. Iniciar Túnel Seguro (ngrok ou Localtunnel) e Servidor FastAPI
# @markdown Insira seu **NGROK_AUTHTOKEN** abaixo (ou salve nos Secrets do Colab com a chave `NGROK_AUTHTOKEN`).
# @markdown *Se deixar em branco, o notebook usará o **Localtunnel** automaticamente como fallback.*

NGROK_AUTHTOKEN = "" # @param {type:"string"}

import nest_asyncio
import uvicorn
import sys
import os
import subprocess
import time
import asyncio

token = NGROK_AUTHTOKEN.strip()
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get('NGROK_AUTHTOKEN') or '').strip()
    except Exception:
        token = os.environ.get("NGROK_AUTHTOKEN", "").strip()

public_url = None

if token:
    print("🔒 NGROK_AUTHTOKEN detectado! Iniciando túnel oficial do ngrok (Alta Estabilidade)...")
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(token)
        ngrok.kill()
        tunnel = ngrok.connect(8000, "http")
        public_url = tunnel.public_url
        if public_url.startswith("http://"):
            public_url = public_url.replace("http://", "https://")

        print("\n" + "=" * 68)
        print("🪸 SERVIDOR CORALSCOP ON-LINE COM NGROK (ALTA VELOCIDADE & ESTÁVEL)!")
        print(f"URL: {public_url}")
        print("Copie a URL acima e cole na interface web local!")
        print("=" * 68 + "\n")
    except Exception as e:
        print(f"❌ Falha ao iniciar ngrok: {e}")
        print("Alternando automaticamente para Localtunnel...")
        token = ""

if not token or not public_url:
    print("🌐 Nenhum NGROK_AUTHTOKEN ativo. Usando Localtunnel...")
    os.system("npm install -g localtunnel > /dev/null 2>&1")
    process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    time.sleep(5)
    for _ in range(5):
        if process.poll() is not None:
            break
        line = process.stdout.readline().strip()
        if "your url is:" in line.lower():
            public_url = line.split("your url is:")[-1].strip()
            break
        time.sleep(1)

    try:
        ip = subprocess.check_output(["curl", "ipv4.icanhazip.com"], text=True).strip()
    except Exception:
        ip = "IP Indisponível"

    print("\n" + "=" * 68)
    print("🪸 SERVIDOR CORALSCOP ON-LINE COM LOCALTUNNEL!")
    print(f"URL: {public_url}")
    print(f"Se pedir Tunnel Password, use o IP: {ip}")
    print("=" * 68 + "\n")

nest_asyncio.apply()
if "_coralscop_uvicorn_server" in globals() and _coralscop_uvicorn_server is not None:
    _coralscop_uvicorn_server.should_exit = True
    time.sleep(0.5)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
_coralscop_uvicorn_server = uvicorn.Server(config)

try:
    loop = asyncio.get_event_loop()
    loop.create_task(_coralscop_uvicorn_server.serve())
    print("🚀 Servidor FastAPI do CoralSCOP rodando em segundo plano no Colab com sucesso!")
except Exception as e:
    print(f"Falha ao acoplar o servidor ao loop ativo: {e}")
